In [1]:
# Import the SparkSession entry point to programmatically configure and initialize Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    # Name the application for identification in the Spark UI and cluster resource manager
    .appName("LakehouseLab")
    # 1. Enable Apache Iceberg SQL extensions (adds custom syntax like MERGE INTO, CREATE OR REPLACE, time travel, etc.)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2")
    # 2. Register a new Iceberg catalog named 'lake' using Spark's SparkCatalog implementation
    .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
    # 3. Set the catalog backend type to 'rest' (communicates with an Iceberg REST catalog server for table metadata)
    .config("spark.sql.catalog.lake.type", "rest")
    # 4. Specify the URI endpoint of the Iceberg REST catalog service container
    .config("spark.sql.catalog.lake.uri", "http://iceberg-rest:8181")
    # 5. Define the root S3 warehouse directory path where Iceberg data files and manifests will be stored
    .config("spark.sql.catalog.lake.warehouse", "s3://warehouse/")
    # 6. Configure the IO implementation class for reading/writing files via S3 APIs
    .config(
        "spark.sql.catalog.lake.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO",
    )
    # 7. Point to the S3-compatible object storage endpoint (e.g., SeaweedFS or MinIO)
    .config("spark.sql.catalog.lake.s3.endpoint", "http://seaweedfs:8333")
    # 8. Force path-style access for S3 requests (required for many local/custom S3-compatible stores instead of virtual-hosted style)
    .config("spark.sql.catalog.lake.s3.path-style-access", "true")
    # 9. Provide the S3 storage authentication credentials (Access Key ID)
    .config("spark.sql.catalog.lake.s3.access-key-id", "lakehouse")
    # 10. Provide the S3 storage authentication credentials (Secret Access Key)
    .config("spark.sql.catalog.lake.s3.secret-access-key", "lakehouse-secret")
    # --- CRITICAL: Iceberg catalog-level S3 region settings ---
    .config("spark.sql.catalog.lake.s3.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint", "http://seaweedfs:8333")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # -----------------------------------------------------------
    # 11. Set 'lake' as the default catalog so tables can be referenced directly without prefixing the catalog name
    .config("spark.sql.defaultCatalog", "lake")
    # Materialize or retrieve the configured SparkSession instance
    .getOrCreate()
)

# Reduce noisy Spark execution logs by setting the root logger level to WARN instead of INFO
spark.sparkContext.setLogLevel("WARN")

# Print the active PySpark runtime version to verify successful initialization
print("Spark:", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-40cbc0bf-b978-4b1d-b731-1e449a7b3504;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
:: resolution report :: resolve 154ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final fr

Spark: 3.5.6


In [2]:
spark.catalog.listCatalogs()

[CatalogMetadata(name='spark_catalog', description=None)]

In [3]:
spark.catalog.listDatabases()

[Database(name='bronze', catalog='lake', description=None, locationUri='s3://warehouse/bronze'),
 Database(name='silver', catalog='lake', description=None, locationUri='s3://warehouse/silver'),
 Database(name='gold', catalog='lake', description=None, locationUri='s3://warehouse/gold')]

In [5]:
spark.read.format('iceberg').load('silver.users').show(5, False)

+---+----------+---------+-----------------------------+--------------------------+--------------------------+-----------+-------------+
|id |first_name|last_name|email                        |created_at                |updated_at                |valid_email|full_name    |
+---+----------+---------+-----------------------------+--------------------------+--------------------------+-----------+-------------+
|1  |Bob       |Haugen   |Elva.Crayton@inea.gov        |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Bob Haugen   |
|2  |Christian |Cox      |D.Barkley@adipiscingduis.info|2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Christian Cox|
|3  |Jere      |Crouch   |Edna@facilisinulla.com       |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Jere Crouch  |
|4  |Ethel     |Farmer   |Deon@lobortisautem.us        |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Ethel Farmer |
|5  |Iona      |Lay      |Theda@quifacili

# **Create Gold Tables with Trino**

- **docker compose exec trino trino :**
  - **SHOW CATALOGS;**
  - **USE iceberg.bronze**
  - **SHOW TABLES**